# Layer 3b: active learning on SentiGOLD where the queried rows are labeled by an LLM (or by gold labels, as the "human" twin)

Colab notebook, GPU runtime, Drive mounted. Saved outputs are cleared; results are in `results/sentigold_3b/`.

Run the cells top to bottom. The last cell runs all six runs (LLM and human annotator x random, margin, entropy) and takes hours.
Each finished round is checkpointed to `MyDrive/llm_results/`, so after a disconnect re-run the cells above it and the last one resumes where it stopped.
The Qwen2.5-7B-Instruct weights are expected in `MyDrive/qwen2.5-7b-instruct` (not in this repo).


In [ ]:
import os
import pickle
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score
from google.colab import files
import os
import torch

!pip install -q transformers accelerate bitsandbytes

from google.colab import drive
drive.mount('/content/drive')

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

CKPT_DIR = "/content/drive/MyDrive/llm_results"
os.makedirs(CKPT_DIR, exist_ok=True)

from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
from datasets import Dataset

In [ ]:
# ===== DROP-IN REPLACEMENT for the BASE_URL / read_csv cell =====
# Produces train_raw, val_raw, test_raw with columns "Data" and "Label" (0=neutral, 1=positive, 2=negative),
# with the column names the rest of the notebook expects.
!pip install -q datasets
from datasets import load_dataset
import pandas as pd
from sklearn.model_selection import train_test_split

SEED = 17
TOTAL = 15700          # subsample size (about 15.7k rows); set None to use everything

ds = load_dataset("SayedShaun/sentigold")
raw = ds["train"].to_pandas()
print("columns:", raw.columns.tolist())
print(raw["Polarity"].value_counts())

# 5 -> 3 classes, same ids as the notebook's LABEL_MAP (neutral=0, positive=1, negative=2)
COLLAPSE = {"SP": 1, "WP": 1, "NU": 0, "WN": 2, "SN": 2}
unknown = set(raw["Polarity"].unique()) - set(COLLAPSE)
assert not unknown, f"Unexpected Polarity values {unknown}: edit COLLAPSE using the counts printed above"

df = pd.DataFrame({"Data": raw["Text"], "Label": raw["Polarity"].map(COLLAPSE)})
if "Domain" in raw.columns:
    df["Domain"] = raw["Domain"]          # extra column, ignored by clean(); handy for later analysis

# pre-apply the notebook's own filters (non-null, 1 < length < 100, no duplicates) so clean() doesn't shrink it much
df = df.dropna(subset=["Data", "Label"])
df["Data"] = df["Data"].astype(str).str.strip()
df = df[(df["Data"].str.len() > 1) & (df["Data"].str.len() < 100)]
df = df.drop_duplicates(subset="Data").reset_index(drop=True)
df["Label"] = df["Label"].astype(int)
print("after filtering:", len(df), dict(df["Label"].value_counts()))

if TOTAL and len(df) > TOTAL:
    df = df.groupby("Label", group_keys=False).apply(
        lambda d: d.sample(frac=TOTAL / len(df), random_state=SEED)).reset_index(drop=True)

# stratified 80 / 10 / 10
train_raw, rest = train_test_split(df, test_size=0.20, stratify=df["Label"], random_state=SEED)
val_raw, test_raw = train_test_split(rest, test_size=0.50, stratify=rest["Label"], random_state=SEED)
train_raw, val_raw, test_raw = [d.reset_index(drop=True) for d in (train_raw, val_raw, test_raw)]

print(train_raw.shape, val_raw.shape, test_raw.shape)
print("label mix (train/val/test):",
      [dict(d["Label"].value_counts().sort_index()) for d in (train_raw, val_raw, test_raw)])

In [ ]:
def clean(df):
    df = df.rename(columns={"Data": "sentence", "Label": "sentiment"}).copy()
    df = df.dropna(subset=["sentence", "sentiment"])
    df["sentence"] = df["sentence"].astype(str).str.strip()
    df = df[(df["sentence"].str.len() > 1) & (df["sentence"].str.len() < 100)]
    df = df.drop_duplicates(subset="sentence").reset_index(drop=True)
    return df[["sentence", "sentiment"]]

pool_source = clean(pd.concat([train_raw, val_raw], ignore_index=True))
test_df     = clean(test_raw)

pool_source = pool_source[~pool_source["sentence"].isin(set(test_df["sentence"]))].reset_index(drop=True)

print("Pool source:", len(pool_source))
print("Locked test:", len(test_df))

In [ ]:
def make_split(pool_source, seed_size=200, seed=17):
    frac = seed_size / len(pool_source)
    seed_train = pool_source.groupby("sentiment", group_keys=False).sample(frac=frac, random_state=seed)
    pool_df = pool_source.drop(seed_train.index).reset_index(drop=True)
    return seed_train.reset_index(drop=True), pool_df

seed_train, pool_df = make_split(pool_source, seed_size=200, seed=17)
print("Seed train:", len(seed_train), dict(seed_train["sentiment"].value_counts()))
print("Pool:", len(pool_df))

In [ ]:
def least_confidence(probs):
    p_max = np.max(probs)
    n = len(probs)
    return -(1 - p_max) * (n / (n - 1))

def margin_sampling(probs):
    top2 = np.sort(probs)[::-1][:2]
    return top2[0] - top2[1]

def ratio_sampling(probs):
    top2 = np.sort(probs)[::-1][:2]
    return top2[0] / (top2[1] + 1e-12)

def entropy_sampling(probs):
    p = np.clip(probs, 1e-12, 1)
    ent = -np.sum(p * np.log2(p)) / np.log2(len(probs))
    return -ent

def random_sampling(probs, rng):
    return rng.random()

STRATEGIES = {
    "least_confidence": least_confidence,
    "margin": margin_sampling,
    "ratio": ratio_sampling,
    "entropy": entropy_sampling,
    "random": random_sampling,
}

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

llm_tokenizer = AutoTokenizer.from_pretrained("/content/drive/MyDrive/qwen2.5-7b-instruct")
llm_model = AutoModelForCausalLM.from_pretrained(
    "/content/drive/MyDrive/qwen2.5-7b-instruct",
    quantization_config=bnb_config,
    device_map="auto",
)

In [ ]:
LABEL_MAP = {"neutral": 0, "positive": 1, "negative": 2}
LABEL_NAMES = {v: k for k, v in LABEL_MAP.items()}

def build_prompt(sentence, examples=None):
    instructions = (
        "You are a sentiment classifier for Bangla text. "
        "Classify the sentiment of the given text as exactly one of: positive, negative, neutral. "
        "Respond with only the single label word, nothing else."
    )
    few_shot_block = ""
    if examples:
        for ex_text, ex_label in examples:
            few_shot_block += f"\nText: {ex_text}\nLabel: {LABEL_NAMES[ex_label]}"
    return f"{instructions}\n{few_shot_block}\n\nText: {sentence}\nLabel:"


In [ ]:
def make_chat_text(sentence, examples=None):
    msgs = [{"role": "user", "content": build_prompt(sentence, examples)}]
    return llm_tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

llm_tokenizer.padding_side = "left"
if llm_tokenizer.pad_token is None:
    llm_tokenizer.pad_token = llm_tokenizer.eos_token

LABEL_WORDS = [LABEL_NAMES[i] for i in range(3)]
LABEL_TOKEN_IDS = [llm_tokenizer.encode(w, add_special_tokens=False)[0] for w in LABEL_WORDS]
assert len(set(LABEL_TOKEN_IDS)) == 3, "label first-tokens collide; use a different label word"

In [ ]:
def atomic_pickle(obj, path):
    """Write to a temp file, then swap in. A disconnect mid-write can't corrupt the real checkpoint."""
    tmp = path + ".tmp"
    with open(tmp, "wb") as f:
        pickle.dump(obj, f)
        f.flush()
        os.fsync(f.fileno())
    os.replace(tmp, path)

In [ ]:
@torch.no_grad()
def llm_annotate_rows(rows_df, examples=None, batch_size=16):
    """Returns (labels, confidences). Replaces the older version that returned labels only."""
    texts = [make_chat_text(s, examples) for s in rows_df["sentence"].tolist()]
    out = np.zeros((len(texts), 3), dtype=np.float32)
    for i in range(0, len(texts), batch_size):
        enc = llm_tokenizer(texts[i:i+batch_size], return_tensors="pt", padding=True).to(llm_model.device)
        logits = llm_model(**enc).logits[:, -1, :]
        out[i:i+batch_size] = torch.softmax(logits[:, LABEL_TOKEN_IDS].float(), dim=-1).cpu().numpy()
    return out.argmax(axis=1), out.max(axis=1)

In [ ]:
MODEL_NAME = "csebuetnlp/banglabert"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_texts(texts, max_length=64):
    return tokenizer(list(texts), truncation=True, padding=True, max_length=max_length, return_tensors="pt")

In [ ]:
# def make_bert_model():
#     model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=3)
#     return model.to(device)

from transformers import set_seed

def make_bert_model(seed):
    set_seed(seed)
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=3)
    return model.to(device)

In [ ]:
def fine_tune_round(train_df, seed, epochs=3, batch_size=16):
    model = make_bert_model(seed)

    train_encodings = tokenize_texts(train_df["sentence"])
    train_dataset = Dataset.from_dict({
        "input_ids": train_encodings["input_ids"],
        "attention_mask": train_encodings["attention_mask"],
        "labels": train_df["sentiment"].tolist(),
    })

    # args = TrainingArguments(
    #     output_dir="/content/bert_tmp",
    #     num_train_epochs=epochs,
    #     per_device_train_batch_size=batch_size,
    #     logging_steps=50,
    #     save_strategy="no",
    #     report_to="none",
    #     seed=seed,
    # )

    args = TrainingArguments(
        output_dir="/content/bert_tmp",
        num_train_epochs=epochs,
        per_device_train_batch_size=batch_size,
        learning_rate=2e-5,      # down from the Trainer default of 5e-5
        # warmup_ratio=0.1,        # ease into training instead of a hard start
        logging_steps=50,
        save_strategy="no",
        report_to="none",
        seed=seed,
    )

    trainer = Trainer(model=model, args=args, train_dataset=train_dataset)
    trainer.train()
    return model

In [ ]:
import torch.nn.functional as F

def predict_probs(model, sentences, batch_size=32):
    model.eval()
    all_probs = []
    with torch.no_grad():
        for i in range(0, len(sentences), batch_size):
            batch = sentences[i:i+batch_size]
            enc = tokenize_texts(batch)
            enc = {k: v.to(device) for k, v in enc.items()}
            logits = model(**enc).logits
            probs = F.softmax(logits, dim=-1).cpu().numpy()
            all_probs.append(probs)
    return np.concatenate(all_probs, axis=0)

In [ ]:
def run_bert_active_learning(seed_train, pool_df, test_df, strategy_name, annotator="llm",
                             n_rounds=15, batch_size=100, seed=17, run_id=None, llm_examples=None):
    assert annotator in ("llm", "human")
    run_id = run_id or f"sentigold_{annotator}_annotator_{strategy_name}"
    state_path = f"{CKPT_DIR}/{run_id}.pkl"
    strategy_fn = STRATEGIES[strategy_name]
    rng = np.random.default_rng(seed)

    train_df = pool = logs = agreement_log = None
    ann_df, start_round = pd.DataFrame(), 0
    if os.path.exists(state_path):
        try:
            with open(state_path, "rb") as f:
                st = pickle.load(f)
            train_df, pool = st["train_df"], st["pool"]
            logs, agreement_log = st["logs"], st["agreement_log"]
            ann_df = st.get("annotations", pd.DataFrame())            # .get: old checkpoints lack these keys
            if st.get("rng_state") is not None:
                rng.bit_generator.state = st["rng_state"]
            start_round = st["round"] + 1
            print(f"Resuming {run_id} from round {start_round}")
        except Exception as e:
            print(f"Checkpoint unreadable ({e!r}) -- starting {run_id} fresh.")
            train_df = None
    if train_df is None:
        train_df, pool = seed_train.copy().reset_index(drop=True), pool_df.copy().reset_index(drop=True)
        logs, agreement_log, ann_df, start_round = [], [], pd.DataFrame(), 0

    for round_i in range(start_round, n_rounds):
        if len(pool) < batch_size:
            break

        model = fine_tune_round(train_df, seed=seed, epochs=3, batch_size=16)
        test_preds = np.argmax(predict_probs(model, test_df["sentence"].tolist()), axis=1)
        f1m = f1_score(test_df["sentiment"].values, test_preds, average="macro")
        acc = accuracy_score(test_df["sentiment"].values, test_preds)
        logs.append({"strategy": strategy_name, "annotator": annotator, "seed": seed, "round": round_i,
                     "train_size": len(train_df), "test_accuracy": acc, "test_f1_macro": f1m})
        print(f"[{run_id}] round {round_i} | train_size={len(train_df)} | test_f1_macro={f1m:.3f} acc={acc:.3f}")

        pool_probs = predict_probs(model, pool["sentence"].tolist())
        if strategy_name == "random":
            scores = np.array([strategy_fn(p, rng) for p in pool_probs])
        else:
            scores = np.array([strategy_fn(p) for p in pool_probs])
        query_idx = np.argsort(scores)[:batch_size]
        queried = pool.iloc[query_idx].copy()

        true_labels = queried["sentiment"].values.copy()          # kept aside for analysis
        if annotator == "llm":
            train_labels, conf = llm_annotate_rows(queried, examples=llm_examples)
        else:
            train_labels, conf = true_labels.copy(), np.ones(len(queried), dtype=np.float32)
        agreement = float((true_labels == train_labels).mean())
        agreement_log.append({"round": round_i, "n_labeled": len(queried), "llm_agreement": agreement})
        print(f"    annotator={annotator} agreement with gold this round: {agreement:.3f}")

        ann_df = pd.concat([ann_df, pd.DataFrame({
            "round": round_i, "sentence": queried["sentence"].values,
            "true_label": true_labels, "train_label": train_labels, "annotator_conf": conf})], ignore_index=True)

        queried["sentiment"] = train_labels                        # BanglaBERT trains on THESE
        pool = pool.drop(pool.index[query_idx]).reset_index(drop=True)
        train_df = pd.concat([train_df, queried], ignore_index=True)

        atomic_pickle({"round": round_i, "train_df": train_df, "pool": pool, "logs": logs,
                       "agreement_log": agreement_log, "annotations": ann_df,
                       "rng_state": rng.bit_generator.state}, state_path)
        del model
        torch.cuda.empty_cache()

    curve = pd.DataFrame(logs)
    curve.to_csv(f"{CKPT_DIR}/{run_id}_curve.csv", index=False)
    ann_df.to_csv(f"{CKPT_DIR}/{run_id}_annotations.csv", index=False)
    return curve, pd.DataFrame(agreement_log), ann_df


In [ ]:
RUNS = [("llm", "random"), ("llm", "margin"), ("llm", "entropy"),
        ("human", "random"), ("human", "margin"), ("human", "entropy")]
results = {}
for annotator, strat in RUNS:
    rid = f"sentigold_{annotator}_annotator_{strat}"      # llm+margin keeps your existing run_id, so it resumes
    results[rid] = run_bert_active_learning(seed_train, pool_df, test_df, strategy_name=strat,
                                            annotator=annotator, n_rounds=15, batch_size=100,
                                            seed=17, run_id=rid, llm_examples=None)
